# 04 Limpieza 02 — Normalizar afiliaciones UNAM y filtrar autores externos

Esta libreta usa la base ya separada artículo–autor–afiliación y los archivos finales de catálogo/diccionario UNAM para generar **un único CSV final**: la base con sólo autores asociados a UNAM y afiliaciones normalizadas.

## 1. Configuración de rutas y columnas canónicas

In [1]:
"""
Fase 02: normalizar afiliaciones UNAM y eliminar filas sin asociación UNAM.

Entrada principal:
- integrado_articulo_autor_afiliacion_unificado_manual.csv

Entradas de referencia:
- diccionario_afiliaciones_unam_final_actualizado.csv
- catalogo_afiliaciones_unam_final_actualizado.csv

Salida única:
- integrado_solo_unam_normalizado.csv

Criterios:
- Mantiene sólo las 14 columnas canónicas en el CSV final.
- Conserva sólo filas donde Afiliacion1 o Afiliacion2 tenga asociación UNAM detectable.
- Normaliza Afiliacion1/Afiliacion2 usando el catálogo/diccionario final.
- Si se detectan dos afiliaciones UNAM distintas, usa Afiliacion1 y Afiliacion2.
- No conserva afiliaciones externas, direcciones, ciudades, códigos postales ni comas en la salida.
- Si sólo se detecta UNAM genérico y no un organismo específico, conserva:
  Universidad Nacional Autonoma de Mexico.
"""
from __future__ import annotations

import csv
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

# ---------------------------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------------------------
RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")

INPUT_CSV = RAIZ_PROYECTO / r"04_Limpieza\00_Separacion_Autor_Afiliacion\integrado_articulo_autor_afiliacion_unificado_manual.csv"
DICT_CSV = RAIZ_PROYECTO / r"04_Limpieza\01_Normalizar_Afiliaciones\diccionario_afiliaciones_unam_final_actualizado.csv"
CATALOG_CSV = RAIZ_PROYECTO / r"04_Limpieza\01_Normalizar_Afiliaciones\catalogo_afiliaciones_unam_final_actualizado.csv"
OUT_DIR = RAIZ_PROYECTO / r"04_Limpieza\01_Normalizar_Afiliaciones"
OUTPUT_CSV = OUT_DIR / "integrado_solo_unam_normalizado.csv"

if not INPUT_CSV.exists():
    INPUT_CSV = Path("/mnt/data/integrado_articulo_autor_afiliacion_unificado_manual.csv")
if not DICT_CSV.exists():
    if Path("/mnt/data/diccionario_afiliaciones_unam_final_actualizado.csv").exists():
        DICT_CSV = Path("/mnt/data/diccionario_afiliaciones_unam_final_actualizado.csv")
    else:
        DICT_CSV = Path("/mnt/data/2_diccionario_afiliaciones_unam_final_actualizado.csv")
if not CATALOG_CSV.exists():
    if Path("/mnt/data/catalogo_afiliaciones_unam_final_actualizado.csv").exists():
        CATALOG_CSV = Path("/mnt/data/catalogo_afiliaciones_unam_final_actualizado.csv")
    else:
        CATALOG_CSV = Path("/mnt/data/2_catalogo_afiliaciones_unam_final_actualizado.csv")
if os.name != "nt" and str(INPUT_CSV).startswith("/mnt/data"):
    # Fallback de prueba en este entorno.
    OUT_DIR = Path("/mnt/data/01_Normalizar_Afiliaciones")
    OUTPUT_CSV = OUT_DIR / "integrado_solo_unam_normalizado.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_COLUMNS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

GENERIC_UNAM = "Universidad Nacional Autonoma de Mexico"



## 2. Reglas del diccionario, señales UNAM y normalización interna

La afiliación escrita en la salida siempre debe venir del catálogo final. La normalización de texto sólo se usa para búsqueda interna.

In [2]:
SUPPLEMENTAL_RULES_RAW = [
    # IIMAS
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán", r"academic unit.*applied mathematics and systems.*(?:yucatan|merida)", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"research institute in applied mathematics and systems", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"applied mathematics and systems research institute", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"institute of applied mathematics and systems research", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"applied mathematics and systems research", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"inst invest mat.*aplicadas.*sistemas", False),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"instituto de investigaciones en matematicas y en sistemas", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"instituto de investigaciones en matematicas aplicadas y sistemas", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", r"investigaciones en matematicas aplicadas.*sistemas", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán", r"unidad academica.*(?:estado de )?yucatan", True),
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán", r"iimas.*(?:unidad|merida|yucatan)", True),

    # Centros
    ("Centro de Ciencias de la Complejidad", r"complexity sciences center", True),
    ("Centro de Ciencias Matemáticas", r"mathematical sciences center", True),
    ("Centro de Ciencias Genómicas", r"(?:center for genomic sciences|genomic sciences center)", True),
    ("Centro de Física Aplicada y Tecnología Avanzada", r"center for applied physics and advanced technology", True),
    ("Centro de Estudios en Computación Avanzada", r"advanced computing studies center|computacion avanzada", True),
    ("Centro de Geociencias", r"institute of geosciences|geosciences center|instituto de geociencias", True),
    ("Centro de Nanociencias y Nanotecnología", r"center for nanosciences and nanotechnology", True),
    ("Centro de Investigaciones de Diseño Industrial", r"industrial design research center", True),
    ("Centro de Investigaciones Interdisciplinarias en Ciencias y Humanidades", r"\bceiich\b|interdisciplinary research center for sciences and humanities", True),

    # FES / ENES
    ("Facultad de Estudios Superiores Cuautitlán", r"faculty of higher studies cuautitlan|\bfesc\b|\bfes\s*c\b|fes cuautitlan", True),
    ("Facultad de Estudios Superiores Cuautitlán", r"facultad de estudios superiores cuatitlan|fes cuatitlan|cuatitlan.*unam", True),
    ("Facultad de Estudios Superiores Aragón", r"faculty of higher studies aragon", True),
    ("Facultad de Estudios Superiores Iztacala", r"faculty of higher studies iztacala", True),
    ("Facultad de Estudios Superiores Zaragoza", r"faculty of higher studies zaragoza", True),
    ("Facultad de Estudios Superiores Acatlán", r"faculty of higher studies acatlan|design and building division|acatlan edomex", True),
    ("Escuela Nacional de Estudios Superiores Unidad Morelia", r"national school of higher studies.*morelia|unam campus morelia", True),
    ("Escuela Nacional de Estudios Superiores Unidad Juriquilla", r"national school of higher studies.*juriquilla", True),
    ("Escuela Nacional de Estudios Superiores Unidad Mérida", r"national school of higher studies.*merida", True),
    ("Escuela Nacional de Estudios Superiores Unidad León", r"national school of higher studies.*leon", True),
    ("Escuela Nacional de Estudios Superiores Unidad León", r"enes.*leon|escuela nacional de estudios superiores.*leon", True),

    # Institutos en inglés
    ("Instituto de Ingeniería", r"institute of engineering|engineering institute", True),
    ("Instituto de Matemáticas", r"institute of mathematics|mathematics institute", True),
    ("Instituto de Matemáticas", r"instituto de mathematicas|instituto de mathemathicas|inst de matematicas", True),
    ("Instituto de Ciencias Aplicadas y Tecnología", r"applied science[s]? and technology|institute of applied science", True),
    ("Instituto de Ciencias Nucleares", r"institute of nuclear sciences", True),
    ("Instituto de Biotecnología", r"institute of biotechnology", True),
    ("Instituto de Física", r"institute of physics|physics institute", True),
    ("Instituto de Física", r"instituto de f sica", True),
    ("Instituto de Ecología", r"institute of ecology", True),
    ("Instituto de Geofísica", r"institute of geophysics|solar radiation department", True),
    ("Instituto de Ciencias de la Atmósfera y Cambio Climático", r"atmospheric sciences and climate change", True),
    ("Instituto de Investigaciones Biomédicas", r"institute of biomedical research", True),
    ("Instituto de Astronomía", r"institute of astronomy|inst astron", True),
    ("Instituto de Astronomía", r"observatorio astronomico nacional", True),
    ("Instituto de Fisiología Celular", r"institute of cellular physiology|cellular physiology", True),
    ("Instituto de Geología", r"institute of geology", True),
    ("Instituto de Energías Renovables", r"institute of renewable energies", True),
    ("Instituto de Biología", r"institute of biology", True),
    ("Instituto de Investigaciones en Materiales", r"materials research institute", True),
    ("Instituto de Química", r"institute of chemistry", True),
    ("Instituto de Neurobiología", r"institute of neurobiology|neurodevelopment research unit", True),
    ("Instituto de Ciencias Físicas", r"institute of physical sciences", True),
    ("Instituto de Ciencias del Mar y Limnología", r"marine sciences and limnology", True),
    ("Instituto de Geografía", r"institute of geography", True),

    # Facultades y departamentos asociados cuando hay señal UNAM directa.
    ("Facultad de Ingeniería", r"faculty of engineering|engineering faculty|school of engineering", True),
    ("Facultad de Ingeniería", r"department of telecommunications|dept of telecommunications|dept telecommunications", True),
    ("Facultad de Ingeniería", r"department of computer engineering|computer engineering department|department of computing", True),
    ("Facultad de Ingeniería", r"department of electrical engineering|division of electrical engineering|electrical engineering", True),
    ("Facultad de Ingeniería", r"signal processing department|dept of mechatronics engineering|mechatronics engineering", True),
    ("Facultad de Ingeniería", r"mechanical engineering|petroleum engineering|geology engineering", True),
    ("Facultad de Ingeniería", r"biomedical systems.*engineering|school of civil engineering|engineering department|department of control and robotics", True),
    ("Facultad de Ingeniería", r"fac(?:ultad)?\s*de\s*ing(?:enieria)?\b|fac ing\b|fac ingenieria|facultad de ingeneria", True),
    ("Facultad de Ingeniería", r"posgrado en ingenieria electrica|posgrado en ingenieria", True),
    ("Facultad de Ingeniería", r"dpto de computacion|departamento de computacion|departamento de energia electrica|departamento de ingenieria industrial", True),
    ("Facultad de Ciencias", r"faculty of science\b|faculty of sciences\b|science faculty\b|sciences faculty\b", True),
    ("Facultad de Ciencias", r"department of mathematics|department of physics|physics department|department of biology", True),
    ("Facultad de Medicina Veterinaria y Zootecnia", r"veterinary medicine", True),
    ("Facultad de Medicina", r"faculty of medicine\b", True),
    ("Facultad de Química", r"faculty of chemistry|school of chemistry|biochemistry department", True),
    ("Facultad de Psicología", r"psychology faculty|faculty of psychology", True),
    ("Facultad de Arquitectura", r"faculty of architecture", True),

    # Coordinaciones y direcciones
    ("Dirección General de Cómputo y de Tecnologías de Información y Comunicación", r"computing and information and communication technologies|information and communication technologies|computo y de tecnologias de informacion y comunicacion", True),
]

PARENT_SUPPRESSION_PAIRS = [
    ("Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán"),
    ("Facultad de Ciencias", "Facultad de Ciencias Políticas y Sociales"),
    ("Facultad de Medicina", "Facultad de Medicina Veterinaria y Zootecnia"),
]

# ---------------------------------------------------------------------------
# Normalización sólo para búsqueda; nunca se escribe en la salida final.
# ---------------------------------------------------------------------------
def strip_accents(value: object) -> str:
    text = "" if value is None else str(value)
    text = text.replace("ı", "i").replace("İ", "I")
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )


def norm_text(value: object) -> str:
    text = strip_accents(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_text(value: object) -> str:
    text = "" if value is None else str(value)
    text = text.replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip(" ;,")


def is_generic_unam(affiliation: str) -> bool:
    return norm_text(affiliation) == norm_text(GENERIC_UNAM)


# ---------------------------------------------------------------------------
# Señales UNAM directas y exclusiones.
# ---------------------------------------------------------------------------
DIRECT_UNAM_PATTERNS = [
    r"\bunam\b",
    r"u\s*n\s*a\s*m",
    r"universidad nacional autonoma de mexico",
    r"universidad nacional autonoma mexico",
    # Variante rara corregida manualmente en la base de entrada.
    r"universidad autonoma de mexico",
    r"national autonomous university of mexico",
    r"national autonomous univ(?:ersity)? of mexico",
    r"national university of mexico",
    r"univ nacl autonoma mex",
    r"univ nac autonoma mex",
    r"univ nacional autonoma de mexico",
    r"nacional autonoma de mexico",
    r"grid\s*9486\s*3",
    r"ror\s*org\s*01tmp8f25",
]
DIRECT_UNAM_RE = [re.compile(pattern) for pattern in DIRECT_UNAM_PATTERNS]

NEGATIVE_PATTERNS = [
    r"afiliacion externa",
    r"afiliacion no unam",
    r"externa no unam",
    r"no unam en la fuente consultada",
    r"no recuperada",
    r"exacta no exportada",
    r"affiliation not",
    r"external affiliation",
    r"not available",
    r"sin afiliacion",
]
NEGATIVE_RE = [re.compile(pattern) for pattern in NEGATIVE_PATTERNS]

# Marcadores externos que ayudan a evitar falsos positivos cuando NO hay señal UNAM directa.
EXTERNAL_SUBSTRINGS = [
    "universidad autonoma metropolitana", "uam",
    "cinvestav", "ipn", "instituto politecnico nacional",
    "tecnologico de monterrey", "instituto tecnologico de nuevo leon", "tecnologico de nuevo leon",
    "universidad veracruzana", "hospital general de mexico", "instituto nacional de",
    "universidad autonoma de queretaro", "uaemex", "universidad autonoma del estado de mexico",
    "universidad autonoma de ciudad juarez", "universidad de guadalajara",
    "universidad de sonora", "universidad de colima", "universidad autonoma de baja california",
    "universidad autonoma de san luis potosi", "universidad autonoma de nuevo leon",
    "universidad de buenos aires", "universidad panamericana",
    "benemerita universidad autonoma de puebla", "pontificia universidad catolica",
    "universidad catolica de chile", "university of", "universite", "universitat",
    "universita", "universidade", "polytechnic", "conacyt", "conahcyt", "inmegen",
    "imss", "issste", "instituto nacional de medicina genomica",
    "instituto nacional de cancerologia", "colegio nacional", "ciesas",
    "center for research and advanced studies", "kanazawa university", "kyoto university",
    "university paris", "paris cite", "cnrs", "lis university aix marseille",
    "spain", "chile", "france", "del conocimiento", "instituto de ingenieria y tecnologia",
    "facultad de ciencias fisico matematicas", "facultad de ciencias de la computacion",
    "facultad de ciencias economicas",
]
EXTERNAL_REGEXES = [
    re.compile(r"universidad autonoma de (?!mexico)"),
]

LOCATION_TOKENS = {
    "mexico", "cdmx", "ciudad", "city", "cu", "coyoacan", "df", "distrito",
    "federal", "mx",
}
TAIL_STOPWORDS = {"de", "del", "la", "las", "el", "los", "y", "e", "en", "of", "the"}


def has_direct_unam_norm(norm_value: str) -> bool:
    return any(pattern.search(norm_value) for pattern in DIRECT_UNAM_RE)


def is_negative_norm(norm_value: str) -> bool:
    return any(pattern.search(norm_value) for pattern in NEGATIVE_RE)


def has_external_marker_norm(norm_value: str, direct_unam: bool) -> bool:
    if direct_unam:
        return False
    return (
        any(marker in norm_value for marker in EXTERNAL_SUBSTRINGS)
        or any(pattern.search(norm_value) for pattern in EXTERNAL_REGEXES)
    )


def clean_tail_tokens(tokens: Iterable[str]) -> List[str]:
    return [token for token in tokens if token not in TAIL_STOPWORDS]


# ---------------------------------------------------------------------------
# Carga de archivos de referencia.
# ---------------------------------------------------------------------------
def read_csv_dicts(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return [{k: clean_text(v) for k, v in row.items()} for row in csv.DictReader(handle)]


def load_catalog(path: Path) -> set[str]:
    rows = read_csv_dicts(path)
    catalog = {row.get("afiliacion", "") for row in rows if row.get("afiliacion", "")}
    return catalog


def load_dictionary(path: Path) -> List[Tuple[str, str, str, str]]:
    rows = read_csv_dicts(path)
    alias_rows: List[Tuple[str, str, str, str]] = []
    for row in rows:
        alias = clean_text(row.get("alias", ""))
        affiliation = clean_text(row.get("afiliacion", ""))
        alias_norm = norm_text(alias)
        if not alias or not affiliation or not alias_norm:
            continue
        # El patrón con espacios permite coincidencia por token sin usar regex costoso.
        alias_rows.append((alias, affiliation, alias_norm, f" {alias_norm} "))

    def priority(item: Tuple[str, str, str, str]) -> Tuple[int, int, int, int]:
        alias, affiliation, alias_norm, _ = item
        return (
            0 if is_generic_unam(affiliation) else 1,
            len(alias_norm.split()),
            len(alias_norm),
            len(alias),
        )

    return sorted(set(alias_rows), key=priority, reverse=True)




def load_supplemental_rules(catalog: set[str]) -> List[Tuple[str, re.Pattern, bool]]:
    """Compila reglas complementarias cuyo destino está permitido por el catálogo."""
    rules: List[Tuple[str, re.Pattern, bool]] = []
    for affiliation, pattern, requires_direct in SUPPLEMENTAL_RULES_RAW:
        if affiliation in catalog:
            rules.append((affiliation, re.compile(pattern), requires_direct))
    return rules


def suppress_parent_units(items: List[Dict[str, object]]) -> List[Dict[str, object]]:
    """Elimina unidades padre cuando aparece una unidad hija más específica."""
    affiliations = {str(item.get("affiliation", "")) for item in items}
    remove = set()
    for parent, child in PARENT_SUPPRESSION_PAIRS:
        if parent in affiliations and child in affiliations:
            remove.add(parent)
    if GENERIC_UNAM in affiliations and len(affiliations) > 1:
        remove.add(GENERIC_UNAM)
    return [item for item in items if str(item.get("affiliation", "")) not in remove]

# ---------------------------------------------------------------------------
# Búsqueda de alias.
# ---------------------------------------------------------------------------
def find_phrase_positions(spaced_norm_text: str, spaced_alias_norm: str) -> List[int]:
    positions: List[int] = []
    start = 0
    while True:
        idx = spaced_norm_text.find(spaced_alias_norm, start)
        if idx < 0:
            break
        # Se resta 1 para compensar el espacio agregado al inicio.
        positions.append(max(idx - 1, 0))
        start = idx + 1
    return positions


def accept_alias_match(
    norm_value: str,
    direct_unam: bool,
    external_marker: bool,
    alias: str,
    affiliation: str,
    alias_norm: str,
    position: int,
) -> Tuple[bool, str]:
    """Decide si un alias detectado debe aceptarse como afiliación UNAM."""
    if is_negative_norm(norm_value):
        return False, "rechazo_placeholder_o_no_unam"

    if direct_unam:
        return True, "unam_directo_alias_diccionario"

    if external_marker:
        return False, "rechazo_marcador_externo_sin_unam_directo"

    # Acrónimos fuertes del diccionario, por ejemplo IIMAS, ICAT, DGTIC.
    if alias.isupper() and len(alias) <= 8:
        return True, "alias_fuerte_sin_unam_directo_acronimo"

    # Programas/posgrados permitidos por el diccionario final.
    if alias_norm.startswith((
        "posgrado en", "licenciatura en", "unidad de posgrado", "ciencias biomedicas",
    )):
        return True, "programa_asociado_por_diccionario"

    # Organismos sin UNAM directo
    organismo_prefixes = (
        "instituto de ", "centro de ", "centro regional de ", "facultad de ",
        "escuela nacional ", "direccion general ",
    )
    if alias_norm.startswith(tuple(norm_text(prefix) for prefix in organismo_prefixes)):
        tail = clean_tail_tokens(norm_value[position + len(alias_norm):].strip().split())
        if not tail or all(token in LOCATION_TOKENS for token in tail[:4]):
            return True, "organismo_diccionario_sin_unam_directo_contexto_limpio"
        return False, "rechazo_organismo_incrustado_en_texto_externo"

    return False, "rechazo_alias_debil_sin_unam_directo"


def normalize_affiliation_text(
    raw_text: str,
    dictionary_rows: List[Tuple[str, str, str, str]],
    supplemental_rules: List[Tuple[str, re.Pattern, bool]],
) -> List[Dict[str, object]]:
    raw_text = clean_text(raw_text)
    norm_value = norm_text(raw_text)

    if not raw_text or is_negative_norm(norm_value):
        return []

    direct_unam = has_direct_unam_norm(norm_value)
    external_marker = has_external_marker_norm(norm_value, direct_unam=direct_unam)
    spaced_norm_value = f" {norm_value} "

    matches: List[Dict[str, object]] = []
    for alias, affiliation, alias_norm, spaced_alias_norm in dictionary_rows:
        for position in find_phrase_positions(spaced_norm_value, spaced_alias_norm):
            accepted, rule = accept_alias_match(
                norm_value=norm_value,
                direct_unam=direct_unam,
                external_marker=external_marker,
                alias=alias,
                affiliation=affiliation,
                alias_norm=alias_norm,
                position=position,
            )
            if accepted:
                matches.append({
                    "position": position,
                    "alias_len": len(alias_norm),
                    "alias": alias,
                    "affiliation": affiliation,
                    "rule": rule,
                })

    # Reglas complementarias en inglés/departamentos.
    for affiliation, pattern, requires_direct in supplemental_rules:
        if requires_direct and not direct_unam:
            continue
        if external_marker and not direct_unam:
            continue
        match = pattern.search(norm_value)
        if match:
            matches.append({
                "position": match.start(),
                "alias_len": match.end() - match.start(),
                "alias": f"__REGLA_COMPLEMENTARIA__:{pattern.pattern}",
                "affiliation": affiliation,
                "rule": "regla_complementaria_diccionario_catalogo",
            })

    # Orden: aparición en el texto, alias más largo primero, específico antes que genérico.
    matches.sort(key=lambda item: (
        int(item["position"]),
        -int(item["alias_len"]),
        1 if is_generic_unam(str(item["affiliation"])) else 0,
    ))

    # Deduplicar afiliaciones finales preservando orden.
    unique: List[Dict[str, object]] = []
    seen = set()
    for item in matches:
        key = norm_text(item["affiliation"])
        if key not in seen:
            seen.add(key)
            unique.append(item)

    unique = suppress_parent_units(unique)
    # Si hay organismos específicos, se elimina el genérico UNAM de esa misma celda.
    if any(not is_generic_unam(str(item["affiliation"])) for item in unique):
        unique = [item for item in unique if not is_generic_unam(str(item["affiliation"]))]

    # Si hay señal UNAM directa, pero no hubo alias específico, conservar el genérico.
    if not unique and direct_unam and not external_marker:
        unique = [{
            "position": 0,
            "alias_len": 0,
            "alias": "__UNAM_DIRECTO_SIN_ALIAS__",
            "affiliation": GENERIC_UNAM,
            "rule": "unam_directo_sin_alias_especifico",
        }]

    return unique


def normalize_row(
    row: Dict[str, str],
    dictionary_rows: List[Tuple[str, str, str, str]],
    supplemental_rules: List[Tuple[str, re.Pattern, bool]],
) -> List[Dict[str, object]]:
    all_matches: List[Dict[str, object]] = []

    for column in ["Afiliacion1", "Afiliacion2"]:
        for match in normalize_affiliation_text(row.get(column, ""), dictionary_rows, supplemental_rules):
            item = dict(match)
            item["column"] = column
            item["raw_text"] = row.get(column, "")
            all_matches.append(item)

    all_matches.sort(key=lambda item: (
        0 if item["column"] == "Afiliacion1" else 1,
        int(item["position"]),
        -int(item["alias_len"]),
        1 if is_generic_unam(str(item["affiliation"])) else 0,
    ))

    unique: List[Dict[str, object]] = []
    seen = set()
    for item in all_matches:
        key = norm_text(item["affiliation"])
        if key not in seen:
            seen.add(key)
            unique.append(item)

    unique = suppress_parent_units(unique)
    # Si una fila tiene organismo específico, no conservar también UNAM genérico.
    if any(not is_generic_unam(str(item["affiliation"])) for item in unique):
        unique = [item for item in unique if not is_generic_unam(str(item["affiliation"]))]

    return unique


## 3. Procesar la base y guardar sólo el CSV final

In [3]:
# ---------------------------------------------------------------------------
# Procesamiento final: sólo genera el CSV limpio normalizado
# ---------------------------------------------------------------------------
def read_input_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != CANONICAL_COLUMNS:
            raise ValueError(
                "El archivo de entrada no tiene exactamente las 14 columnas canónicas. "
                f"Columnas recibidas: {reader.fieldnames}"
            )
        return [{k: clean_text(v) for k, v in row.items()} for row in reader]


def write_csv(path: Path, rows: List[Dict[str, object]], columns: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def process() -> Dict[str, object]:
    catalog = load_catalog(CATALOG_CSV)
    dictionary_rows = load_dictionary(DICT_CSV)
    supplemental_rules = load_supplemental_rules(catalog)
    input_rows = read_input_rows(INPUT_CSV)

    final_rows: List[Dict[str, object]] = []
    rule_counter: Counter = Counter()
    normalized_aff_counter: Counter = Counter()
    rows_with_more_than_two = 0

    for row in input_rows:
        matches = normalize_row(row, dictionary_rows, supplemental_rules)

        if not matches:
            # No hay asociación UNAM en Afiliacion1 ni Afiliacion2: se elimina la fila.
            continue

        detected_affiliations = [str(item["affiliation"]) for item in matches]

        if len(detected_affiliations) > 2:
            rows_with_more_than_two += 1

        aff1_norm = detected_affiliations[0] if len(detected_affiliations) >= 1 else ""
        aff2_norm = detected_affiliations[1] if len(detected_affiliations) >= 2 else ""

        out = {column: row.get(column, "") for column in CANONICAL_COLUMNS}
        out["Afiliacion1"] = aff1_norm
        out["Afiliacion2"] = aff2_norm
        final_rows.append(out)

        for item in matches:
            rule_counter[str(item.get("rule", ""))] += 1

        for aff in [aff1_norm, aff2_norm]:
            if aff:
                normalized_aff_counter[aff] += 1

    # Validaciones de salida final.
    final_aff_values = [
        str(row.get(column, ""))
        for row in final_rows
        for column in ["Afiliacion1", "Afiliacion2"]
        if str(row.get(column, "")).strip()
    ]

    affiliations_with_commas = [aff for aff in final_aff_values if "," in aff]
    affiliations_not_in_catalog = sorted({aff for aff in final_aff_values if aff not in catalog})
    rows_empty_final = sum(
        1 for row in final_rows
        if not str(row.get("Afiliacion1", "")).strip()
        and not str(row.get("Afiliacion2", "")).strip()
    )
    rows_same_aff1_aff2 = sum(
        1 for row in final_rows
        if str(row.get("Afiliacion1", "")).strip()
        and norm_text(row.get("Afiliacion1", "")) == norm_text(row.get("Afiliacion2", ""))
    )

    if affiliations_with_commas:
        raise ValueError(
            "Hay afiliaciones normalizadas con coma. Ejemplos: "
            + str(affiliations_with_commas[:10])
        )
    if affiliations_not_in_catalog:
        raise ValueError(
            "Hay afiliaciones finales fuera del catálogo. Ejemplos: "
            + str(affiliations_not_in_catalog[:20])
        )
    if rows_empty_final:
        raise ValueError(f"Hay {rows_empty_final} filas finales sin afiliación UNAM normalizada.")
    if rows_same_aff1_aff2:
        raise ValueError(f"Hay {rows_same_aff1_aff2} filas con Afiliacion1 igual a Afiliacion2.")

    write_csv(OUTPUT_CSV, final_rows, CANONICAL_COLUMNS)

    resumen = {
        "archivo_entrada": str(INPUT_CSV),
        "diccionario_usado": str(DICT_CSV),
        "catalogo_usado": str(CATALOG_CSV),
        "csv_final": str(OUTPUT_CSV),
        "filas_originales": len(input_rows),
        "filas_conservadas_solo_unam": len(final_rows),
        "filas_eliminadas_sin_unam": len(input_rows) - len(final_rows),
        "articulos_originales": len({row.get("indice", "") for row in input_rows}),
        "articulos_con_autor_unam": len({row.get("indice", "") for row in final_rows}),
        "filas_con_afiliacion2_normalizada": sum(1 for row in final_rows if str(row.get("Afiliacion2", "")).strip()),
        "afiliaciones_normalizadas_distintas": len(set(final_aff_values)),
        "filas_con_mas_de_dos_afiliaciones_unam_detectadas": rows_with_more_than_two,
        "afiliaciones_finales_con_coma": len(affiliations_with_commas),
        "afiliaciones_finales_fuera_catalogo": len(affiliations_not_in_catalog),
        "alias_diccionario": len(dictionary_rows),
        "reglas_complementarias_activas": len(supplemental_rules),
        "afiliaciones_catalogo": len(catalog),
        "top_afiliaciones_normalizadas": dict(normalized_aff_counter.most_common(20)),
        "distribucion_reglas": dict(rule_counter),
    }

    return resumen


resumen = process()

print("Proceso terminado correctamente.")
print(f"Archivo final guardado en: {resumen['csv_final']}")
print(f"Filas originales: {resumen['filas_originales']:,}")
print(f"Filas conservadas sólo UNAM: {resumen['filas_conservadas_solo_unam']:,}")
print(f"Filas eliminadas sin UNAM: {resumen['filas_eliminadas_sin_unam']:,}")
print(f"Artículos con autor UNAM: {resumen['articulos_con_autor_unam']:,}")
print(f"Afiliaciones normalizadas distintas: {resumen['afiliaciones_normalizadas_distintas']:,}")
print(f"Filas con Afiliacion2 normalizada: {resumen['filas_con_afiliacion2_normalizada']:,}")

print("\nTop afiliaciones normalizadas:")
for afiliacion, frecuencia in list(resumen["top_afiliaciones_normalizadas"].items())[:15]:
    print(f"- {afiliacion}: {frecuencia}")


Proceso terminado correctamente.
Archivo final guardado en: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\01_Normalizar_Afiliaciones\integrado_solo_unam_normalizado.csv
Filas originales: 8,940
Filas conservadas sólo UNAM: 3,239
Filas eliminadas sin UNAM: 5,701
Artículos con autor UNAM: 1,459
Afiliaciones normalizadas distintas: 48
Filas con Afiliacion2 normalizada: 82

Top afiliaciones normalizadas:
- Universidad Nacional Autonoma de Mexico: 1243
- Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas: 468
- Facultad de Ingeniería: 374
- Facultad de Ciencias: 172
- Facultad de Estudios Superiores Aragón: 123
- Instituto de Matemáticas: 122
- Instituto de Ingeniería: 109
- Instituto de Física: 90
- Facultad de Estudios Superiores Cuautitlán: 77
- Instituto de Ciencias Aplicadas y Tecnología: 64
- Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán: 59
- Centro de Estudios en Computación Avanzada: 42
- Instituto de Ciencias Físicas: 41
- Institu